## Double Header Data

One shortcoming of using Statcast’s API is that it does not distinguish doubleheaders, making it impossible to reliably differentiate multiple games played by the same teams on the same day. While this limitation does not materially affect batter–pitcher–ballpark combinations in the hierarchical model, accurately identifying individual games is important for understanding game-to-game trends, which in turn informs the construction of more informative priors. Therefore, it is necessary to ensure that each game is uniquely accounted for when analyzing trends across a season.

To address this issue, I will first collect game-level data from ESPN, which explicitly identifies doubleheader games. I will then use Baseball-Reference data to cross-reference starting pitchers for each game. Because pitchers are uniquely associated with individual games within a doubleheader, this information allows me to correctly assign game numbers and match each observation to the appropriate Statcast data in the Game Processing notebook.

In [1]:
import sys
from pathlib import Path

# Find the repo root by searching upwards for src/hrmodel
here = Path.cwd()
for p in (here, *here.parents):
    if (p / "src" / "hrmodel").exists():
        REPO = p
        break
else:
    raise RuntimeError("Could not find repo root containing src/hrmodel")

# Put src/ on sys.path (front so it wins)
sys.path.insert(0, str(REPO / "src"))


In [2]:
# -----------------------------
# Core libraries
# -----------------------------
import re
import time
import ast
from datetime import date
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd

# -----------------------------
# Web scraping (requests + BeautifulSoup)
# -----------------------------
import requests
from bs4 import BeautifulSoup, Comment
from io import StringIO

# -----------------------------
# Progress + notebook display
# -----------------------------
from tqdm.auto import tqdm
from IPython.display import display, HTML

# -----------------------------
# Custom
# -----------------------------
from hrmodel.utils.preprocessing.dh_proprocessing import (
    espn_doubleheaders_textparse,
    normalize_doubleheader_dates,
    drop_postponed_doubleheaders,
    replace_away_home_with_abbr,
    expand_double_headers,
    sort_doubleheaders,
    build_team_name_to_abbrev_map,
    add_target_team_codes,
    add_bref_boxscore_urls,
    collect_pitchers_for_season,
)

In [3]:
from IPython.core.magic import register_cell_magic

@register_cell_magic
def skip(line, cell):
    """No-op this cell."""
    return

## ESPN

Below, I construct a cleaned doubleheader reference table using ESPN. For each season (2022–2025), I:

1. Scrape and parse ESPN’s doubleheaders page into a structured DataFrame.
2. Normalize ESPN’s date strings into standardized datetime formats.
3. Drop postponed entries to retain only valid, played doubleheaders.

The resulting season-specific DataFrames (`dh_22`–`dh_25`) serve as a reference for identifying doubleheader days and matchups. These tables are used downstream to disambiguate same-day games in Statcast data and to assign the correct game index within each doubleheader.


### 2025

In [5]:
#%%skip

url = "https://www.espn.com/mlb/stats/doubleheaders/_/year/2025"
dh_25 = espn_doubleheaders_textparse(url)
dh_25 = normalize_doubleheader_dates(dh_25, year=2025)
dh_25 = drop_postponed_doubleheaders(dh_25)
dh_25.head(10)

,DATE,TEAMS,GAME 1,GAME 2
0,2025-04-06,St. Louis at Boston,"BOS,\n5-4","BOS,\n18-7"
1,2025-04-20,Washington at Colorado,"COL,\n3-1","WSH,\n3-2"
2,2025-04-24,Colorado at Kansas City,"KC,\n7-4","KC,\n6-2"
3,2025-04-26,Boston at Cleveland,"CLE,\n5-4","BOS,\n7-3"
4,2025-04-26,Baltimore at Detroit,"DET,\n4-3","DET,\n6-2"
5,2025-04-27,Toronto at NY Yankees,"NYY,\n5-1","NYY,\n11-2"
6,2025-04-30,St. Louis at Cincinnati,"STL,\n9-1","STL,\n6-0"
7,2025-05-04,NY Mets at St. Louis,"STL,\n6-5","STL,\n5-4"
8,2025-05-06,Cleveland at Washington,"WSH,\n10-9","CLE,\n9-1"
9,2025-05-08,Detroit at Colorado,"DET,\n10-2","DET,\n11-1"


In [6]:
#%%skip

url = "https://www.espn.com/mlb/stats/doubleheaders/_/year/2024"
dh_24 = espn_doubleheaders_textparse(url)
dh_24 = normalize_doubleheader_dates(dh_24, year=2024)
dh_24 = drop_postponed_doubleheaders(dh_24)
dh_24.head(10)

,DATE,TEAMS,GAME 1,GAME 2
0,2024-04-04,Detroit at NY Mets,"NYM,\n2-1","DET,\n6-3"
1,2024-04-13,NY Yankees at Cleveland,"NYY,\n3-2","NYY,\n8-2"
2,2024-04-13,Minnesota at Detroit,"MIN,\n4-1","MIN,\n11-5"
3,2024-04-17,Kansas City at Chicago Sox,"KC,\n4-2","CHW,\n2-1"
4,2024-04-20,Miami at Chicago Cubs,"CHC,\n5-3","MIA,\n3-2"
5,2024-04-21,Seattle at Colorado,"SEA,\n10-2","COL,\n2-1"
6,2024-04-30,St. Louis at Detroit,"DET,\n11-6","STL,\n2-1"
7,2024-05-08,Texas at Athletics,"ATH,\n9-4","TEX,\n12-11"
8,2024-05-14,Washington at Chicago Sox,"WSH,\n6-3","CHW,\n4-0"
9,2024-05-20,San Diego at Atlanta,"ATL,\n3-0","SD,\n6-5"


### 2023

In [7]:
#%%skip

url = "https://www.espn.com/mlb/stats/doubleheaders/_/year/2023"
dh_23 = espn_doubleheaders_textparse(url)
dh_23 = normalize_doubleheader_dates(dh_23, year=2023)
dh_23 = drop_postponed_doubleheaders(dh_23)
dh_23.head(10)

,DATE,TEAMS,GAME 1,GAME 2
0,2023-04-18,Philadelphia at Chicago Sox,"CHW,\n3-0","PHI,\n7-4"
1,2023-04-18,Cleveland at Detroit,"DET,\n4-3","DET,\n1-0"
2,2023-04-22,Miami at Cleveland,"MIA,\n3-2","MIA,\n6-1"
3,2023-04-29,Baltimore at Detroit,"BAL,\n6-4","DET,\n7-4"
4,2023-04-29,Pittsburgh at Washington,"PIT,\n16-1","PIT,\n6-3"
5,2023-05-01,Atlanta at NY Mets,"NYM,\n5-3","ATL,\n9-8"
6,2023-05-03,NY Mets at Detroit,"DET,\n8-1","DET,\n6-5"
7,2023-05-21,Cleveland at NY Mets,"NYM,\n2-1","NYM,\n5-4"
8,2023-06-03,Tampa Bay at Boston,"TB,\n4-2","BOS,\n8-5"
9,2023-06-08,Chicago Sox at NY Yankees,"CHW,\n6-5","NYY,\n3-0"


### 2022

In [8]:
#%%skip

url = "https://www.espn.com/mlb/stats/doubleheaders/_/year/2022"
dh_22 = espn_doubleheaders_textparse(url)
dh_22 = normalize_doubleheader_dates(dh_22, year=2022)
dh_22.head(10)

,DATE,TEAMS,GAME 1,GAME 2
0,2022-04-19,Arizona at Washington,"WSH,\n6-1","WSH,\n1-0"
1,2022-04-19,San Francisco at NY Mets,"NYM,\n5-4","NYM,\n3-1"
2,2022-04-20,Chicago Sox at Cleveland,"CLE,\n11-1","CLE,\n2-1"
3,2022-04-23,Colorado at Detroit,"DET,\n13-0","COL,\n3-2"
4,2022-05-03,Atlanta at NY Mets,"NYM,\n5-4","NYM,\n3-0"
5,2022-05-04,San Diego at Cleveland,"SD,\n5-4","CLE,\n6-5"
6,2022-05-04,Pittsburgh at Detroit,"DET,\n3-2","PIT,\n7-2"
7,2022-05-07,Toronto at Cleveland,"TOR,\n8-3","CLE,\n8-2"
8,2022-05-07,LA Dodgers at Chicago Cubs,"LAD,\n7-0","LAD,\n6-2"
9,2022-05-07,Pittsburgh at Cincinnati,"CIN,\n9-2","PIT,\n8-5"


### Team abbreviations

Next, I standardize team identifiers by converting ESPN’s matchup strings into the same team abbreviations used in Statcast. I implement a small set of helper functions to:

1. Define a short-name → abbreviation mapping (`team_to_abbr_short`) that matches Statcast’s team codes.
2. Normalize team labels with regex-based rules to handle common naming variants and formatting differences.
3. Parse the `TEAMS` field (formatted as `Away at Home`) to extract `away_team` and `home_team`, then map each to the corresponding Statcast-style abbreviation.

This produces standardized `away_team` and `home_team` columns across `dh_22`–`dh_25`, enabling reliable merges between the ESPN doubleheader reference tables and Statcast game-level data.


In [10]:
#%%skip

dh_25 = replace_away_home_with_abbr(dh_25)  
dh_24 = replace_away_home_with_abbr(dh_24)  
dh_23 = replace_away_home_with_abbr(dh_23)  
dh_22 = replace_away_home_with_abbr(dh_22)  

display(HTML("<h4>Season 2025</h4>")); display(dh_25.head(10))
display(HTML("<h4>Season 2024</h4>")); display(dh_24.head(10))
display(HTML("<h4>Season 2023</h4>")); display(dh_23.head(10))
display(HTML("<h4>Season 2022</h4>")); display(dh_22.head(10))

,DATE,TEAMS,GAME 1,GAME 2,away_team,home_team
0,2025-04-06,St. Louis at Boston,"BOS,\n5-4","BOS,\n18-7",STL,BOS
1,2025-04-20,Washington at Colorado,"COL,\n3-1","WSH,\n3-2",WSH,COL
2,2025-04-24,Colorado at Kansas City,"KC,\n7-4","KC,\n6-2",COL,KC
3,2025-04-26,Boston at Cleveland,"CLE,\n5-4","BOS,\n7-3",BOS,CLE
4,2025-04-26,Baltimore at Detroit,"DET,\n4-3","DET,\n6-2",BAL,DET
5,2025-04-27,Toronto at NY Yankees,"NYY,\n5-1","NYY,\n11-2",TOR,NYY
6,2025-04-30,St. Louis at Cincinnati,"STL,\n9-1","STL,\n6-0",STL,CIN
7,2025-05-04,NY Mets at St. Louis,"STL,\n6-5","STL,\n5-4",NYM,STL
8,2025-05-06,Cleveland at Washington,"WSH,\n10-9","CLE,\n9-1",CLE,WSH
9,2025-05-08,Detroit at Colorado,"DET,\n10-2","DET,\n11-1",DET,COL


,DATE,TEAMS,GAME 1,GAME 2,away_team,home_team
0,2024-04-04,Detroit at NY Mets,"NYM,\n2-1","DET,\n6-3",DET,NYM
1,2024-04-13,NY Yankees at Cleveland,"NYY,\n3-2","NYY,\n8-2",NYY,CLE
2,2024-04-13,Minnesota at Detroit,"MIN,\n4-1","MIN,\n11-5",MIN,DET
3,2024-04-17,Kansas City at Chicago Sox,"KC,\n4-2","CHW,\n2-1",KC,CWS
4,2024-04-20,Miami at Chicago Cubs,"CHC,\n5-3","MIA,\n3-2",MIA,CHC
5,2024-04-21,Seattle at Colorado,"SEA,\n10-2","COL,\n2-1",SEA,COL
6,2024-04-30,St. Louis at Detroit,"DET,\n11-6","STL,\n2-1",STL,DET
7,2024-05-08,Texas at Athletics,"ATH,\n9-4","TEX,\n12-11",TEX,ATH
8,2024-05-14,Washington at Chicago Sox,"WSH,\n6-3","CHW,\n4-0",WSH,CWS
9,2024-05-20,San Diego at Atlanta,"ATL,\n3-0","SD,\n6-5",SD,ATL


,DATE,TEAMS,GAME 1,GAME 2,away_team,home_team
0,2023-04-18,Philadelphia at Chicago Sox,"CHW,\n3-0","PHI,\n7-4",PHI,CWS
1,2023-04-18,Cleveland at Detroit,"DET,\n4-3","DET,\n1-0",CLE,DET
2,2023-04-22,Miami at Cleveland,"MIA,\n3-2","MIA,\n6-1",MIA,CLE
3,2023-04-29,Baltimore at Detroit,"BAL,\n6-4","DET,\n7-4",BAL,DET
4,2023-04-29,Pittsburgh at Washington,"PIT,\n16-1","PIT,\n6-3",PIT,WSH
5,2023-05-01,Atlanta at NY Mets,"NYM,\n5-3","ATL,\n9-8",ATL,NYM
6,2023-05-03,NY Mets at Detroit,"DET,\n8-1","DET,\n6-5",NYM,DET
7,2023-05-21,Cleveland at NY Mets,"NYM,\n2-1","NYM,\n5-4",CLE,NYM
8,2023-06-03,Tampa Bay at Boston,"TB,\n4-2","BOS,\n8-5",TB,BOS
9,2023-06-08,Chicago Sox at NY Yankees,"CHW,\n6-5","NYY,\n3-0",CWS,NYY


,DATE,TEAMS,GAME 1,GAME 2,away_team,home_team
0,2022-04-19,Arizona at Washington,"WSH,\n6-1","WSH,\n1-0",AZ,WSH
1,2022-04-19,San Francisco at NY Mets,"NYM,\n5-4","NYM,\n3-1",SF,NYM
2,2022-04-20,Chicago Sox at Cleveland,"CLE,\n11-1","CLE,\n2-1",CWS,CLE
3,2022-04-23,Colorado at Detroit,"DET,\n13-0","COL,\n3-2",COL,DET
4,2022-05-03,Atlanta at NY Mets,"NYM,\n5-4","NYM,\n3-0",ATL,NYM
5,2022-05-04,San Diego at Cleveland,"SD,\n5-4","CLE,\n6-5",SD,CLE
6,2022-05-04,Pittsburgh at Detroit,"DET,\n3-2","PIT,\n7-2",PIT,DET
7,2022-05-07,Toronto at Cleveland,"TOR,\n8-3","CLE,\n8-2",TOR,CLE
8,2022-05-07,LA Dodgers at Chicago Cubs,"LAD,\n7-0","LAD,\n6-2",LAD,CHC
9,2022-05-07,Pittsburgh at Cincinnati,"CIN,\n9-2","PIT,\n8-5",PIT,CIN


### Expanding doubleheaders to one row per game

Next, I convert the doubleheader tables from wide format (separate `GAME 1` and `GAME 2` columns) into a long, game-level format with one row per game. Using `expand_double_headers`, I:

1. Validate that the required columns are present (including `DATE`, `TEAMS`, `away_team`, `home_team`, `GAME 1`, and `GAME 2`).
2. Reshape the data by melting `GAME 1` and `GAME 2` into a single `raw_result` column, with a corresponding `game_label`.
3. Extract `game_number` (1 or 2) from `game_label` and sort by `DATE` and `game_number` for consistent ordering.

This produces a standardized game-level reference table that can be used downstream to disambiguate same-day games and support merges with Statcast data.


**TODO**: Combine `sort_doubleheaders` with `expand_double_headers`.

In [11]:
%%skip

def expand_double_headers(df):
    """
    Take a double-header DataFrame (with GAME 1 / GAME 2 columns)
    and return a long DataFrame with one row per game.

    Columns required:
      - 'DATE'
      - 'TEAMS'
      - 'away_team'
      - 'home_team'
      - 'GAME 1'
      - 'GAME 2'

    Output columns include:
      - DATE
      - TEAMS
      - away_team
      - home_team
      - game_label   (GAME 1 / GAME 2)
      - raw_result   (e.g., 'BOS,\\n5-4')
      - game_number  (1 or 2)
    """
    required_cols = ["DATE", "TEAMS", "away_team", "home_team", "GAME 1", "GAME 2"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"expand_double_headers: missing required columns: {missing}")

    long_df = df.melt(
        id_vars=["DATE", "TEAMS", "away_team", "home_team"],
        value_vars=["GAME 1", "GAME 2"],
        var_name="game_label",
        value_name="raw_result"
    )

    # Extract game number: "GAME 1" -> 1, "GAME 2" -> 2
    long_df["game_number"] = long_df["game_label"].str.extract(r"(\d+)").astype(int)

    # Sort by date then game number
    long_df = (
        long_df
        .sort_values(["DATE", "game_number"])
        .reset_index(drop=True)
    )

    return long_df

In [12]:
#%%skip

dh_25 = expand_double_headers(dh_25)
dh_24 = expand_double_headers(dh_24)
dh_23 = expand_double_headers(dh_23)
dh_22 = expand_double_headers(dh_22)

display(HTML("<h4>Season 2025</h4>")); display(dh_25.head(10))
display(HTML("<h4>Season 2024</h4>")); display(dh_24.head(10))
display(HTML("<h4>Season 2023</h4>")); display(dh_23.head(10))
display(HTML("<h4>Season 2022</h4>")); display(dh_22.head(10))

,DATE,TEAMS,away_team,home_team,game_label,raw_result,game_number
0,2025-04-06,St. Louis at Boston,STL,BOS,GAME 1,"BOS,\n5-4",1
1,2025-04-06,St. Louis at Boston,STL,BOS,GAME 2,"BOS,\n18-7",2
2,2025-04-20,Washington at Colorado,WSH,COL,GAME 1,"COL,\n3-1",1
3,2025-04-20,Washington at Colorado,WSH,COL,GAME 2,"WSH,\n3-2",2
4,2025-04-24,Colorado at Kansas City,COL,KC,GAME 1,"KC,\n7-4",1
5,2025-04-24,Colorado at Kansas City,COL,KC,GAME 2,"KC,\n6-2",2
6,2025-04-26,Boston at Cleveland,BOS,CLE,GAME 1,"CLE,\n5-4",1
7,2025-04-26,Baltimore at Detroit,BAL,DET,GAME 1,"DET,\n4-3",1
8,2025-04-26,Boston at Cleveland,BOS,CLE,GAME 2,"BOS,\n7-3",2
9,2025-04-26,Baltimore at Detroit,BAL,DET,GAME 2,"DET,\n6-2",2


,DATE,TEAMS,away_team,home_team,game_label,raw_result,game_number
0,2024-04-04,Detroit at NY Mets,DET,NYM,GAME 1,"NYM,\n2-1",1
1,2024-04-04,Detroit at NY Mets,DET,NYM,GAME 2,"DET,\n6-3",2
2,2024-04-13,NY Yankees at Cleveland,NYY,CLE,GAME 1,"NYY,\n3-2",1
3,2024-04-13,Minnesota at Detroit,MIN,DET,GAME 1,"MIN,\n4-1",1
4,2024-04-13,NY Yankees at Cleveland,NYY,CLE,GAME 2,"NYY,\n8-2",2
5,2024-04-13,Minnesota at Detroit,MIN,DET,GAME 2,"MIN,\n11-5",2
6,2024-04-17,Kansas City at Chicago Sox,KC,CWS,GAME 1,"KC,\n4-2",1
7,2024-04-17,Kansas City at Chicago Sox,KC,CWS,GAME 2,"CHW,\n2-1",2
8,2024-04-20,Miami at Chicago Cubs,MIA,CHC,GAME 1,"CHC,\n5-3",1
9,2024-04-20,Miami at Chicago Cubs,MIA,CHC,GAME 2,"MIA,\n3-2",2


,DATE,TEAMS,away_team,home_team,game_label,raw_result,game_number
0,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,GAME 1,"CHW,\n3-0",1
1,2023-04-18,Cleveland at Detroit,CLE,DET,GAME 1,"DET,\n4-3",1
2,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,GAME 2,"PHI,\n7-4",2
3,2023-04-18,Cleveland at Detroit,CLE,DET,GAME 2,"DET,\n1-0",2
4,2023-04-22,Miami at Cleveland,MIA,CLE,GAME 1,"MIA,\n3-2",1
5,2023-04-22,Miami at Cleveland,MIA,CLE,GAME 2,"MIA,\n6-1",2
6,2023-04-29,Baltimore at Detroit,BAL,DET,GAME 1,"BAL,\n6-4",1
7,2023-04-29,Pittsburgh at Washington,PIT,WSH,GAME 1,"PIT,\n16-1",1
8,2023-04-29,Baltimore at Detroit,BAL,DET,GAME 2,"DET,\n7-4",2
9,2023-04-29,Pittsburgh at Washington,PIT,WSH,GAME 2,"PIT,\n6-3",2


,DATE,TEAMS,away_team,home_team,game_label,raw_result,game_number
0,2022-04-19,Arizona at Washington,AZ,WSH,GAME 1,"WSH,\n6-1",1
1,2022-04-19,San Francisco at NY Mets,SF,NYM,GAME 1,"NYM,\n5-4",1
2,2022-04-19,Arizona at Washington,AZ,WSH,GAME 2,"WSH,\n1-0",2
3,2022-04-19,San Francisco at NY Mets,SF,NYM,GAME 2,"NYM,\n3-1",2
4,2022-04-20,Chicago Sox at Cleveland,CWS,CLE,GAME 1,"CLE,\n11-1",1
5,2022-04-20,Chicago Sox at Cleveland,CWS,CLE,GAME 2,"CLE,\n2-1",2
6,2022-04-23,Colorado at Detroit,COL,DET,GAME 1,"DET,\n13-0",1
7,2022-04-23,Colorado at Detroit,COL,DET,GAME 2,"COL,\n3-2",2
8,2022-05-03,Atlanta at NY Mets,ATL,NYM,GAME 1,"NYM,\n5-4",1
9,2022-05-03,Atlanta at NY Mets,ATL,NYM,GAME 2,"NYM,\n3-0",2


### Creating Home and Away Scores

**NOTE**: No Longer created home and away scores (TO DELETE)

In [13]:
%%skip

def sort_doubleheaders(df: pd.DataFrame) -> pd.DataFrame:
    """
    Sort a doubleheader dataframe so the two games for each matchup/day are adjacent
    and in consistent order (using gamePk as the tie-breaker).
    """
    out = df.copy()
    out["DATE"] = pd.to_datetime(out["DATE"])
    return out.sort_values(["DATE", "away_team", "home_team"]).reset_index(drop=True)


In [14]:
#%%skip

dh_25 = sort_doubleheaders(dh_25)
dh_24 = sort_doubleheaders(dh_24)
dh_23 = sort_doubleheaders(dh_23)
dh_22 = sort_doubleheaders(dh_22)

display(HTML("<h4>Season 2025</h4>")); display(dh_25.head(10))
display(HTML("<h4>Season 2024</h4>")); display(dh_24.head(10))
display(HTML("<h4>Season 2023</h4>")); display(dh_23.head(10))
display(HTML("<h4>Season 2022</h4>")); display(dh_22.head(10))

,DATE,TEAMS,away_team,home_team,game_label,raw_result,game_number
0,2025-04-06,St. Louis at Boston,STL,BOS,GAME 1,"BOS,\n5-4",1
1,2025-04-06,St. Louis at Boston,STL,BOS,GAME 2,"BOS,\n18-7",2
2,2025-04-20,Washington at Colorado,WSH,COL,GAME 1,"COL,\n3-1",1
3,2025-04-20,Washington at Colorado,WSH,COL,GAME 2,"WSH,\n3-2",2
4,2025-04-24,Colorado at Kansas City,COL,KC,GAME 1,"KC,\n7-4",1
5,2025-04-24,Colorado at Kansas City,COL,KC,GAME 2,"KC,\n6-2",2
6,2025-04-26,Baltimore at Detroit,BAL,DET,GAME 1,"DET,\n4-3",1
7,2025-04-26,Baltimore at Detroit,BAL,DET,GAME 2,"DET,\n6-2",2
8,2025-04-26,Boston at Cleveland,BOS,CLE,GAME 1,"CLE,\n5-4",1
9,2025-04-26,Boston at Cleveland,BOS,CLE,GAME 2,"BOS,\n7-3",2


,DATE,TEAMS,away_team,home_team,game_label,raw_result,game_number
0,2024-04-04,Detroit at NY Mets,DET,NYM,GAME 1,"NYM,\n2-1",1
1,2024-04-04,Detroit at NY Mets,DET,NYM,GAME 2,"DET,\n6-3",2
2,2024-04-13,Minnesota at Detroit,MIN,DET,GAME 1,"MIN,\n4-1",1
3,2024-04-13,Minnesota at Detroit,MIN,DET,GAME 2,"MIN,\n11-5",2
4,2024-04-13,NY Yankees at Cleveland,NYY,CLE,GAME 1,"NYY,\n3-2",1
5,2024-04-13,NY Yankees at Cleveland,NYY,CLE,GAME 2,"NYY,\n8-2",2
6,2024-04-17,Kansas City at Chicago Sox,KC,CWS,GAME 1,"KC,\n4-2",1
7,2024-04-17,Kansas City at Chicago Sox,KC,CWS,GAME 2,"CHW,\n2-1",2
8,2024-04-20,Miami at Chicago Cubs,MIA,CHC,GAME 1,"CHC,\n5-3",1
9,2024-04-20,Miami at Chicago Cubs,MIA,CHC,GAME 2,"MIA,\n3-2",2


,DATE,TEAMS,away_team,home_team,game_label,raw_result,game_number
0,2023-04-18,Cleveland at Detroit,CLE,DET,GAME 1,"DET,\n4-3",1
1,2023-04-18,Cleveland at Detroit,CLE,DET,GAME 2,"DET,\n1-0",2
2,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,GAME 1,"CHW,\n3-0",1
3,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,GAME 2,"PHI,\n7-4",2
4,2023-04-22,Miami at Cleveland,MIA,CLE,GAME 1,"MIA,\n3-2",1
5,2023-04-22,Miami at Cleveland,MIA,CLE,GAME 2,"MIA,\n6-1",2
6,2023-04-29,Baltimore at Detroit,BAL,DET,GAME 1,"BAL,\n6-4",1
7,2023-04-29,Baltimore at Detroit,BAL,DET,GAME 2,"DET,\n7-4",2
8,2023-04-29,Pittsburgh at Washington,PIT,WSH,GAME 1,"PIT,\n16-1",1
9,2023-04-29,Pittsburgh at Washington,PIT,WSH,GAME 2,"PIT,\n6-3",2


,DATE,TEAMS,away_team,home_team,game_label,raw_result,game_number
0,2022-04-19,Arizona at Washington,AZ,WSH,GAME 1,"WSH,\n6-1",1
1,2022-04-19,Arizona at Washington,AZ,WSH,GAME 2,"WSH,\n1-0",2
2,2022-04-19,San Francisco at NY Mets,SF,NYM,GAME 1,"NYM,\n5-4",1
3,2022-04-19,San Francisco at NY Mets,SF,NYM,GAME 2,"NYM,\n3-1",2
4,2022-04-20,Chicago Sox at Cleveland,CWS,CLE,GAME 1,"CLE,\n11-1",1
5,2022-04-20,Chicago Sox at Cleveland,CWS,CLE,GAME 2,"CLE,\n2-1",2
6,2022-04-23,Colorado at Detroit,COL,DET,GAME 1,"DET,\n13-0",1
7,2022-04-23,Colorado at Detroit,COL,DET,GAME 2,"COL,\n3-2",2
8,2022-05-03,Atlanta at NY Mets,ATL,NYM,GAME 1,"NYM,\n5-4",1
9,2022-05-03,Atlanta at NY Mets,ATL,NYM,GAME 2,"NYM,\n3-0",2


### Abbreviation

In [15]:
%%skip

def build_team_name_to_abbrev_map(df: pd.DataFrame, teams_col: str = "TEAMS") -> dict:
    out = df.copy()

    # split "Arizona at Washington" (or "Arizona vs Washington")
    parts = out[teams_col].astype(str).str.split(r"\s+(?:at|vs)\s+", expand=True)

    out["away_name"] = parts[0].str.strip()
    out["home_name"] = parts[1].str.strip()

    # build mapping from names -> abbreviations
    away_map = dict(zip(out["away_name"], out["away_team"]))
    home_map = dict(zip(out["home_name"], out["home_team"]))

    # merge (they should agree if a name appears in both roles)
    name_to_abbrev = {**away_map, **home_map}
    return name_to_abbrev


In [16]:
#%%skip

map_25 = build_team_name_to_abbrev_map(dh_25)
map_24 = build_team_name_to_abbrev_map(dh_24)
map_23 = build_team_name_to_abbrev_map(dh_23)
map_22 = build_team_name_to_abbrev_map(dh_22)

combined_map = {}
for df in [dh_22, dh_23, dh_24, dh_25]:
    combined_map.update(build_team_name_to_abbrev_map(df))

combined_map

{'Arizona': 'AZ',
 'San Francisco': 'SF',
 'Chicago Sox': 'CWS',
 'Colorado': 'COL',
 'Atlanta': 'ATL',
 'Pittsburgh': 'PIT',
 'San Diego': 'SD',
 'LA Dodgers': 'LAD',
 'Toronto': 'TOR',
 'Kansas City': 'KC',
 'NY Mets': 'NYM',
 'Texas': 'TEX',
 'LA Angels': 'LAA',
 'St. Louis': 'STL',
 'Baltimore': 'BAL',
 'Milwaukee': 'MIL',
 'Minnesota': 'MIN',
 'Miami': 'MIA',
 'Philadelphia': 'PHI',
 'NY Yankees': 'NYY',
 'Tampa Bay': 'TB',
 'Cleveland': 'CLE',
 'Detroit': 'DET',
 'Seattle': 'SEA',
 'Chicago Cubs': 'CHC',
 'Cincinnati': 'CIN',
 'Washington': 'WSH',
 'Athletics': 'ATH',
 'Boston': 'BOS',
 'Houston': 'HOU'}

## Baseball Reference

### Abbreviating Team Names

Arizona Diamondbacks: ARI

Atlanta Braves: ATL

Baltimore Orioles: BAL

Boston Red Sox: BOS

Chicago Cubs: CHN

Chicago White Sox: CHA

Cincinnati Reds: CIN

Cleveland Guardians: CLE

Colorado Rockies: COL

Detroit Tigers: DET

Houston Astros: HOU

Kansas City Royals: KCA

Los Angeles Angels: ANA

Los Angeles Dodgers: LAN

Miami Marlins: MIA

Milwaukee Brewers: MIL

Minnesota Twins: MIN
 
New York Mets: NYN

New York Yankees: NYA

Oakland Athletics: OAK or ATH

Philadelphia Phillies: PHI

Pittsburgh Pirates: PIT

San Diego Padres: SDN

San Francisco Giants: SFN

Seattle Mariners: SEA

St. Louis Cardinals: SLN

Tampa Bay Rays: TBA

Texas Rangers: TEX

Toronto Blue Jays: TOR

Washington Nationals: WAS

In [17]:
%%skip

def add_target_team_codes(df: pd.DataFrame, season: int,
                          away_col: str = "away_team",
                          home_col: str = "home_team",
                          new_away_col: str = "away_team_code",
                          new_home_col: str = "home_team_code") -> pd.DataFrame:
    """
    Adds target team-code columns (BRef/Retrosheet-style) to a DH dataframe.
    Oakland caveat: uses ATH in 2025, OAK otherwise.
    """
    out = df.copy()

    mapping = ABBR_TO_TARGET_2025 if season == 2025 else ABBR_TO_TARGET_NON_2025

    out[new_away_col] = out[away_col].map(mapping)
    out[new_home_col] = out[home_col].map(mapping)

    # Optional safety check: surface any unmapped abbreviations
    unmapped = sorted(set(pd.concat([out[away_col], out[home_col]])).difference(mapping.keys()))
    if unmapped:
        raise ValueError(f"Unmapped team abbreviations found for season {season}: {unmapped}")

    return out

In [18]:
%%skip

# Abbreviations to Target
ABBR_TO_TARGET_2025 = {
    "AZ":  "ARI",
    "ATL": "ATL",
    "BAL": "BAL",
    "BOS": "BOS",
    "CHC": "CHN",   # Cubs
    "CWS": "CHA",   # White Sox
    "CIN": "CIN",
    "CLE": "CLE",
    "COL": "COL",
    "DET": "DET",
    "HOU": "HOU",
    "KC":  "KCA",
    "LAA": "ANA",
    "LAD": "LAN",
    "MIA": "MIA",
    "MIL": "MIL",
    "MIN": "MIN",
    "NYM": "NYN",
    "NYY": "NYA",
    "ATH": "ATH",   # 2025 ONLY
    "PHI": "PHI",
    "PIT": "PIT",
    "SD":  "SDN",
    "SF":  "SFN",
    "SEA": "SEA",
    "STL": "SLN",
    "TB":  "TBA",
    "TEX": "TEX",
    "TOR": "TOR",
    "WSH": "WAS",
}

# For non-2025 seasons, Oakland should be OAK
ABBR_TO_TARGET_NON_2025 = ABBR_TO_TARGET_2025.copy()
ABBR_TO_TARGET_NON_2025["ATH"] = "OAK"

In [19]:
#%%skip

# --- apply to your dataframes ---
dh_25 = add_target_team_codes(dh_25, season=2025)
dh_24 = add_target_team_codes(dh_24, season=2024)
dh_23 = add_target_team_codes(dh_23, season=2023)
dh_22 = add_target_team_codes(dh_22, season=2022)

display(HTML("<h4>Season 2025</h4>")); display(dh_25.head(10))
display(HTML("<h4>Season 2024</h4>")); display(dh_24.head(10))
display(HTML("<h4>Season 2023</h4>")); display(dh_23.head(10))
display(HTML("<h4>Season 2022</h4>")); display(dh_22.head(10))

,DATE,TEAMS,away_team,home_team,game_label,raw_result,game_number,away_team_code,home_team_code
0,2025-04-06,St. Louis at Boston,STL,BOS,GAME 1,"BOS,\n5-4",1,SLN,BOS
1,2025-04-06,St. Louis at Boston,STL,BOS,GAME 2,"BOS,\n18-7",2,SLN,BOS
2,2025-04-20,Washington at Colorado,WSH,COL,GAME 1,"COL,\n3-1",1,WAS,COL
3,2025-04-20,Washington at Colorado,WSH,COL,GAME 2,"WSH,\n3-2",2,WAS,COL
4,2025-04-24,Colorado at Kansas City,COL,KC,GAME 1,"KC,\n7-4",1,COL,KCA
5,2025-04-24,Colorado at Kansas City,COL,KC,GAME 2,"KC,\n6-2",2,COL,KCA
6,2025-04-26,Baltimore at Detroit,BAL,DET,GAME 1,"DET,\n4-3",1,BAL,DET
7,2025-04-26,Baltimore at Detroit,BAL,DET,GAME 2,"DET,\n6-2",2,BAL,DET
8,2025-04-26,Boston at Cleveland,BOS,CLE,GAME 1,"CLE,\n5-4",1,BOS,CLE
9,2025-04-26,Boston at Cleveland,BOS,CLE,GAME 2,"BOS,\n7-3",2,BOS,CLE


,DATE,TEAMS,away_team,home_team,game_label,raw_result,game_number,away_team_code,home_team_code
0,2024-04-04,Detroit at NY Mets,DET,NYM,GAME 1,"NYM,\n2-1",1,DET,NYN
1,2024-04-04,Detroit at NY Mets,DET,NYM,GAME 2,"DET,\n6-3",2,DET,NYN
2,2024-04-13,Minnesota at Detroit,MIN,DET,GAME 1,"MIN,\n4-1",1,MIN,DET
3,2024-04-13,Minnesota at Detroit,MIN,DET,GAME 2,"MIN,\n11-5",2,MIN,DET
4,2024-04-13,NY Yankees at Cleveland,NYY,CLE,GAME 1,"NYY,\n3-2",1,NYA,CLE
5,2024-04-13,NY Yankees at Cleveland,NYY,CLE,GAME 2,"NYY,\n8-2",2,NYA,CLE
6,2024-04-17,Kansas City at Chicago Sox,KC,CWS,GAME 1,"KC,\n4-2",1,KCA,CHA
7,2024-04-17,Kansas City at Chicago Sox,KC,CWS,GAME 2,"CHW,\n2-1",2,KCA,CHA
8,2024-04-20,Miami at Chicago Cubs,MIA,CHC,GAME 1,"CHC,\n5-3",1,MIA,CHN
9,2024-04-20,Miami at Chicago Cubs,MIA,CHC,GAME 2,"MIA,\n3-2",2,MIA,CHN


,DATE,TEAMS,away_team,home_team,game_label,raw_result,game_number,away_team_code,home_team_code
0,2023-04-18,Cleveland at Detroit,CLE,DET,GAME 1,"DET,\n4-3",1,CLE,DET
1,2023-04-18,Cleveland at Detroit,CLE,DET,GAME 2,"DET,\n1-0",2,CLE,DET
2,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,GAME 1,"CHW,\n3-0",1,PHI,CHA
3,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,GAME 2,"PHI,\n7-4",2,PHI,CHA
4,2023-04-22,Miami at Cleveland,MIA,CLE,GAME 1,"MIA,\n3-2",1,MIA,CLE
5,2023-04-22,Miami at Cleveland,MIA,CLE,GAME 2,"MIA,\n6-1",2,MIA,CLE
6,2023-04-29,Baltimore at Detroit,BAL,DET,GAME 1,"BAL,\n6-4",1,BAL,DET
7,2023-04-29,Baltimore at Detroit,BAL,DET,GAME 2,"DET,\n7-4",2,BAL,DET
8,2023-04-29,Pittsburgh at Washington,PIT,WSH,GAME 1,"PIT,\n16-1",1,PIT,WAS
9,2023-04-29,Pittsburgh at Washington,PIT,WSH,GAME 2,"PIT,\n6-3",2,PIT,WAS


,DATE,TEAMS,away_team,home_team,game_label,raw_result,game_number,away_team_code,home_team_code
0,2022-04-19,Arizona at Washington,AZ,WSH,GAME 1,"WSH,\n6-1",1,ARI,WAS
1,2022-04-19,Arizona at Washington,AZ,WSH,GAME 2,"WSH,\n1-0",2,ARI,WAS
2,2022-04-19,San Francisco at NY Mets,SF,NYM,GAME 1,"NYM,\n5-4",1,SFN,NYN
3,2022-04-19,San Francisco at NY Mets,SF,NYM,GAME 2,"NYM,\n3-1",2,SFN,NYN
4,2022-04-20,Chicago Sox at Cleveland,CWS,CLE,GAME 1,"CLE,\n11-1",1,CHA,CLE
5,2022-04-20,Chicago Sox at Cleveland,CWS,CLE,GAME 2,"CLE,\n2-1",2,CHA,CLE
6,2022-04-23,Colorado at Detroit,COL,DET,GAME 1,"DET,\n13-0",1,COL,DET
7,2022-04-23,Colorado at Detroit,COL,DET,GAME 2,"COL,\n3-2",2,COL,DET
8,2022-05-03,Atlanta at NY Mets,ATL,NYM,GAME 1,"NYM,\n5-4",1,ATL,NYN
9,2022-05-03,Atlanta at NY Mets,ATL,NYM,GAME 2,"NYM,\n3-0",2,ATL,NYN


### Web Scraping Baseball Reference

In [20]:
%%skip

def add_bref_boxscore_urls(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # ensure datetime
    out["DATE"] = pd.to_datetime(out["DATE"])

    # create YYYYMMDD string
    out["bref_date"] = out["DATE"].dt.strftime("%Y%m%d")

    # ensure game_number is 1/2 as a string
    out["bref_game_number"] = out["game_number"].astype(int).astype(str)

    # build url
    out["bref_url"] = (
        "https://www.baseball-reference.com/boxes/"
        + out["home_team_code"] + "/"
        + out["home_team_code"] + out["bref_date"] + out["bref_game_number"]
        + ".shtml"
    )

    return out

In [21]:
%%skip

# ---------- parsing helpers ----------

def table_to_df_from_comments(soup, table_id: str) -> pd.DataFrame:
    for c in soup.find_all(string=lambda t: isinstance(t, Comment)):
        commented = BeautifulSoup(c, "html.parser")
        table = commented.find("table", {"id": table_id})
        if table is not None:
            return pd.read_html(StringIO(str(table)))[0]
    raise ValueError(f"Table id '{table_id}' not found in comments.")

def fix_mojibake(s: str) -> str:
    try:
        return s.encode("latin1").decode("utf-8")
    except UnicodeError:
        return s

def clean_pitcher_name(raw: str) -> str:
    # "Zack Wheeler, W (1-1)" -> "Zack Wheeler"
    raw = fix_mojibake(raw)
    return re.sub(r",\s*[A-Z]{1,3}\s*\([^)]*\)\s*$", "", raw).strip()

def pitchers_who_pitched(pitch_df: pd.DataFrame) -> list[str]:
    name_col = pitch_df.columns[0]
    s = pitch_df[name_col].astype(str).str.strip()
    s = s[~s.str.contains("Team Totals", case=False, na=False)]
    s = s[s.ne("")]

    # Clean + preserve order, de-dupe just in case
    out, seen = [], set()
    for x in s.tolist():
        x = clean_pitcher_name(x)
        if x and x not in seen:
            seen.add(x)
            out.append(x)
    return out

def bref_team_slug(team_name: str) -> str:
    # "St. Louis" -> "StLouis", "NY Yankees" -> "NYYankees"
    return re.sub(r"[^A-Za-z0-9]", "", str(team_name))

# ---------- BRef fetch + extraction ----------

def build_bref_boxscore_url(home_team_code: str, date, game_number: int | str) -> str:
    """
    home_team_code like 'BOS', date like '2025-04-06' or Timestamp, game_number 1/2
    """
    dt = pd.to_datetime(date)
    yyyymmdd = dt.strftime("%Y%m%d")
    g = str(int(game_number))
    return f"https://www.baseball-reference.com/boxes/{home_team_code}/{home_team_code}{yyyymmdd}{g}.shtml"

def fetch_soup(url: str, session=None, sleep_s: float = 1.5) -> BeautifulSoup:
    sess = session or requests.Session()
    headers = {"User-Agent": "personal-research (contact: you@example.com)"}
    resp = sess.get(url, headers=headers, timeout=30)
    resp.raise_for_status()
    resp.encoding = "utf-8"
    time.sleep(sleep_s)  # polite
    return BeautifulSoup(resp.text, "html.parser")

def get_pitching_table_ids_from_row(row) -> tuple[str, str]:
    """
    Returns (away_pitching_id, home_pitching_id) using away_team_code/home_team_code
    (e.g., SLN/BOS) -> (StLouisCardinalspitching, BostonRedSoxpitching)
    """
    away_slug = ABBR_TO_BREF_SLUG[row["away_team_code"]]
    home_slug = ABBR_TO_BREF_SLUG[row["home_team_code"]]
    return f"{away_slug}pitching", f"{home_slug}pitching"

def get_pitchers_for_game(row, session=None) -> dict:
    """
    row must have: DATE, TEAMS, game_number, home_team_code, away_team_code
    Returns dict with date + url + home/away pitchers
    """
    url = build_bref_boxscore_url(
        home_team_code=row["home_team_code"],
        date=row["DATE"],
        game_number=row["game_number"],
    )

    soup = fetch_soup(url, session=session)

    away_pid, home_pid = get_pitching_table_ids_from_row(row)

    away_pit = table_to_df_from_comments(soup, away_pid)
    home_pit = table_to_df_from_comments(soup, home_pid)

    return {
        "date": pd.to_datetime(row["DATE"]).date(),
        "teams": row.get("TEAMS", None),

        # add abbreviations here
        "away_team": row["away_team"],
        "home_team": row["home_team"],

        "game_number": int(row["game_number"]),
        "bref_url": url,
        "away_pitchers": pitchers_who_pitched(away_pit),
        "home_pitchers": pitchers_who_pitched(home_pit),
    }

In [22]:
%%skip

def collect_pitchers_for_season(dh_df: pd.DataFrame, sleep_s: float = 1.5) -> pd.DataFrame:
    """
    Loops over a DH dataframe (one row per game) and collects BRef pitchers for each game.
    Returns a DataFrame with one row per game (pitchers are list columns).
    """
    out_rows = []
    sess = requests.Session()

    dh_iter = dh_df.reset_index(drop=True).iterrows()

    for i, row in tqdm(dh_iter, total=len(dh_df), desc="Scraping BRef boxscores"):
        try:
            res = get_pitchers_for_game(row, session=sess)
            out_rows.append(res)
        except Exception as e:
            out_rows.append({
                "date": pd.to_datetime(row["DATE"]).date(),
                "teams": row.get("TEAMS", None),
                "away_team": row.get("away_team", None),
                "home_team": row.get("home_team", None),
                "game_number": int(row.get("game_number", 0)) if pd.notna(row.get("game_number", None)) else None,
                "bref_url": build_bref_boxscore_url(row["home_team_code"], row["DATE"], row["game_number"]),
                "away_pitchers": None,
                "home_pitchers": None,
                "error": str(e),
            })

        time.sleep(sleep_s)

    return pd.DataFrame(out_rows)


In [23]:
%%skip

ABBR_TO_BREF_SLUG = {
    "ARI": "ArizonaDiamondbacks",
    "ATL": "AtlantaBraves",
    "BAL": "BaltimoreOrioles",
    "BOS": "BostonRedSox",
    "CHN": "ChicagoCubs",
    "CHA": "ChicagoWhiteSox",
    "CIN": "CincinnatiReds",
    "CLE": "ClevelandGuardians",
    "COL": "ColoradoRockies",
    "DET": "DetroitTigers",
    "HOU": "HoustonAstros",
    "KCA": "KansasCityRoyals",
    "ANA": "LosAngelesAngels",
    "LAN": "LosAngelesDodgers",
    "MIA": "MiamiMarlins",
    "MIL": "MilwaukeeBrewers",
    "MIN": "MinnesotaTwins",
    "NYN": "NewYorkMets",
    "NYA": "NewYorkYankees",
    "OAK": "OaklandAthletics",
    "ATH": "OaklandAthletics",   # table slug is still OaklandAthletics
    "PHI": "PhiladelphiaPhillies",
    "PIT": "PittsburghPirates",
    "SDN": "SanDiegoPadres",
    "SFN": "SanFranciscoGiants",
    "SEA": "SeattleMariners",
    "SLN": "StLouisCardinals",
    "TBA": "TampaBayRays",
    "TEX": "TexasRangers",
    "TOR": "TorontoBlueJays",
    "WAS": "WashingtonNationals",
}


In [24]:
#%%skip

pitchers_25 = collect_pitchers_for_season(dh_25, sleep_s=1.5)
pitchers_24 = collect_pitchers_for_season(dh_24, sleep_s=1.5)
pitchers_23 = collect_pitchers_for_season(dh_23, sleep_s=1.5)
pitchers_22 = collect_pitchers_for_season(dh_22, sleep_s=1.5)

display(HTML("<h4>Season 2025</h4>")); display(pitchers_25.head(10))
display(HTML("<h4>Season 2024</h4>")); display(pitchers_24.head(10))
display(HTML("<h4>Season 2023</h4>")); display(pitchers_23.head(10))
display(HTML("<h4>Season 2022</h4>")); display(pitchers_22.head(10))


Scraping BRef boxscores:   9%|█▋                 | 5/56 [00:15<02:41,  3.17s/it]


KeyboardInterrupt: 

### Saving to Temporary Folder in Doubleheaders

In [17]:
%%skip

# Collect existing pitchers DataFrames into a dict automatically
pitchers_by_year = {
    year: df for year in range(2022, 2026)
    if (df := globals().get(f"pitchers_{str(year)[-2:]}")) is not None
}

# Create subfolder: data/double_headers/pitchers
outdir = Path("data") / "double_headers" / "pitchers"
outdir.mkdir(parents=True, exist_ok=True)

for year, df in pitchers_by_year.items():
    df.to_csv(outdir / f"pitchers_{year}.csv", index=False, encoding="utf-8")
    print(f"[saved] {outdir / f'pitchers_{year}.csv'}")


In [18]:
base_dir = Path("data") / "double_headers" / "pitchers"

pitchers_by_year = {}

for year in range(2022, 2026):
    fp = base_dir / f"pitchers_{year}.csv"
    if fp.exists():
        df = pd.read_csv(fp)

        # restore list columns (saved as strings in CSV)
        for col in ["away_pitchers", "home_pitchers"]:
            if col in df.columns:
                df[col] = df[col].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else x)

        pitchers_by_year[year] = df
    else:
        print(f"[missing] {fp}")

pitchers_22 = pitchers_by_year.get(2022)
pitchers_23 = pitchers_by_year.get(2023)
pitchers_24 = pitchers_by_year.get(2024)
pitchers_25 = pitchers_by_year.get(2025)

In [19]:
display(HTML("<h4>Season 2025</h4>")); display(pitchers_25.head(10))
display(HTML("<h4>Season 2024</h4>")); display(pitchers_24.head(10))
display(HTML("<h4>Season 2023</h4>")); display(pitchers_23.head(10))
display(HTML("<h4>Season 2022</h4>")); display(pitchers_22.head(10))

,date,teams,away_team,home_team,game_number,bref_url,away_pitchers,home_pitchers
0,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,"[Andre Pallante, Kyle Leahy, JoJo Romero, Phil...","[Sean Newcomb, Greg Weissert, Justin Wilson, J..."
1,2025-04-06,St. Louis at Boston,STL,BOS,2,https://www.baseball-reference.com/boxes/BOS/B...,"[Miles Mikolas, Gordon Graceffo, John King, Ch...","[Hunter Dobbins, Brennan Bernardino, Cooper Cr..."
2,2025-04-20,Washington at Colorado,WSH,COL,1,https://www.baseball-reference.com/boxes/COL/C...,"[Jake Irvin, Jose A. Ferrer, Kyle Finnegan]","[Kyle Freeland, Angel Chivilli, Jimmy Herget, ..."
3,2025-04-20,Washington at Colorado,WSH,COL,2,https://www.baseball-reference.com/boxes/COL/C...,"[Brad Lord, Colin Poche, Jackson Rutledge, Edu...","[Antonio Senzatela, Jake Bird, Zach Agnos, Tyl..."
4,2025-04-24,Colorado at Kansas City,COL,KC,1,https://www.baseball-reference.com/boxes/KCA/K...,"[Germán Márquez, Jake Bird, Jimmy Herget]","[Cole Ragans, Angel Zerpa, Steven Cruz, Lucas ..."
5,2025-04-24,Colorado at Kansas City,COL,KC,2,https://www.baseball-reference.com/boxes/KCA/K...,"[Chase Dollander, Jaden Hill, Zach Agnos, Juan...","[Michael Lorenzen, John Schreiber, Daniel Lync..."
6,2025-04-26,Baltimore at Detroit,BAL,DET,1,https://www.baseball-reference.com/boxes/DET/D...,"[Brandon Young, Bryan Baker, Cionel Pérez, Mat...","[Casey Mize, Brenan Hanifee, Tyler Holton, Wil..."
7,2025-04-26,Baltimore at Detroit,BAL,DET,2,https://www.baseball-reference.com/boxes/DET/D...,"[Keegan Akin, Charlie Morton, Seranthony Domín...","[Keider Montero, Brant Hurter, Sean Guenther, ..."
8,2025-04-26,Boston at Cleveland,BOS,CLE,1,https://www.baseball-reference.com/boxes/CLE/C...,"[Tanner Houck, Brennan Bernardino, Greg Weisse...","[Ben Lively, Tim Herrin, Hunter Gaddis, Emmanu..."
9,2025-04-26,Boston at Cleveland,BOS,CLE,2,https://www.baseball-reference.com/boxes/CLE/C...,"[Walker Buehler, Justin Wilson, Justin Slaten,...","[Doug Nikhazy, Kolby Allard]"


,date,teams,away_team,home_team,game_number,bref_url,away_pitchers,home_pitchers
0,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,"[Casey Mize, Joey Wentz, Alex Lange, Andrew Ch...","[Adrian Houser, Brooks Raley, Drew Smith, Jake..."
1,2024-04-04,Detroit at NY Mets,DET,NYM,2,https://www.baseball-reference.com/boxes/NYN/N...,"[Matt Manning, Tyler Holton, Alex Faedo, BS (1)]","[José Buttó, Reed Garrett]"
2,2024-04-13,Minnesota at Detroit,MIN,DET,1,https://www.baseball-reference.com/boxes/DET/D...,"[Joe Ryan, Steven Okert, Griffin Jax, Brock St...","[Kenta Maeda, Tyler Holton, Shelby Miller, Jas..."
3,2024-04-13,Minnesota at Detroit,MIN,DET,2,https://www.baseball-reference.com/boxes/DET/D...,"[Simeon Woods Richardson, Kody Funderburk, Col...","[Matt Manning, Joey Wentz]"
4,2024-04-13,NY Yankees at Cleveland,NYY,CLE,1,https://www.baseball-reference.com/boxes/CLE/C...,"[Clarke Schmidt, Caleb Ferguson, Ian Hamilton,...","[Carlos Carrasco, Nick Sandlin, Eli Morgan, Ti..."
5,2024-04-13,NY Yankees at Cleveland,NYY,CLE,2,https://www.baseball-reference.com/boxes/CLE/C...,"[Cody Poteet, Dennis Santana, Ron Marinaccio]","[Triston McKenzie, Tyler Beede, Cade Smith, We..."
6,2024-04-17,Kansas City at Chicago Sox,KC,CWS,1,https://www.baseball-reference.com/boxes/CHA/C...,"[Brady Singer, Will Smith, Nick Anderson, John...","[Jonathan Cannon, Jordan Leasure, Steven Wilso..."
7,2024-04-17,Kansas City at Chicago Sox,KC,CWS,2,https://www.baseball-reference.com/boxes/CHA/C...,"[Michael Wacha, Angel Zerpa, Chris Stratton]","[Erick Fedde, Tanner Banks, Deivi García]"
8,2024-04-20,Miami at Chicago Cubs,MIA,CHC,1,https://www.baseball-reference.com/boxes/CHN/C...,"[Jesús Luzardo, Calvin Faucher, Tanner Scott]","[Javier Assad, Luke Little, Yency Almonte, Mar..."
9,2024-04-20,Miami at Chicago Cubs,MIA,CHC,2,https://www.baseball-reference.com/boxes/CHN/C...,"[Roddery Muñoz, Anthony Bender, BS (1), Bryan ...","[Shota Imanaga, Ben Brown, Héctor Neris]"


,date,teams,away_team,home_team,game_number,bref_url,away_pitchers,home_pitchers
0,2023-04-18,Cleveland at Detroit,CLE,DET,1,https://www.baseball-reference.com/boxes/DET/D...,"[Hunter Gaddis, Eli Morgan, Nick Sandlin, Jame...","[Matthew Boyd, Mason Englert, Alex Lange]"
1,2023-04-18,Cleveland at Detroit,CLE,DET,2,https://www.baseball-reference.com/boxes/DET/D...,"[Peyton Battenfield, Xzavion Curry]","[Eduardo Rodríguez, Jason Foley]"
2,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,1,https://www.baseball-reference.com/boxes/CHA/C...,"[Zack Wheeler, Gregory Soto, Craig Kimbrel, Se...","[Lance Lynn, Jimmy Lambert, Gregory Santos, Ja..."
3,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,2,https://www.baseball-reference.com/boxes/CHA/C...,"[Bailey Falter, Luis Ortiz]","[Lucas Giolito, Kendall Graveman, Aaron Bummer..."
4,2023-04-22,Miami at Cleveland,MIA,CLE,1,https://www.baseball-reference.com/boxes/CLE/C...,"[Devin Smeltzer, Andrew Nardi, Huascar Brazobá...","[Shane Bieber, Nick Sandlin, Tim Herrin, Enyel..."
5,2023-04-22,Miami at Cleveland,MIA,CLE,2,https://www.baseball-reference.com/boxes/CLE/C...,"[Braxton Garrett, Tanner Scott, Dylan Floro, A...","[Zach Plesac, James Karinchak, Eli Morgan, Emm..."
6,2023-04-29,Baltimore at Detroit,BAL,DET,1,https://www.baseball-reference.com/boxes/DET/D...,"[Dean Kremer, DL Hall]","[Eduardo Rodríguez, Mason Englert, Alex Lange]"
7,2023-04-29,Baltimore at Detroit,BAL,DET,2,https://www.baseball-reference.com/boxes/DET/D...,"[Grayson Rodriguez, Keegan Akin, Mike Baumann,...","[Matthew Boyd, José Cisnero, Jason Foley, Will..."
8,2023-04-29,Pittsburgh at Washington,PIT,WSH,1,https://www.baseball-reference.com/boxes/WAS/W...,"[Rich Hill, Robert Stephenson, Colin Holderman...","[Patrick Corbin, Carl Edwards Jr., Thaddeus Wa..."
9,2023-04-29,Pittsburgh at Washington,PIT,WSH,2,https://www.baseball-reference.com/boxes/WAS/W...,"[Vince Velasquez, Cody Bolton, Yohan Ramírez]","[Chad Kuhl, Jordan Weems, Hobie Harris, Hunter..."


,date,teams,away_team,home_team,game_number,bref_url,away_pitchers,home_pitchers,error
0,2022-04-19,Arizona at Washington,AZ,WSH,1,https://www.baseball-reference.com/boxes/WAS/W...,"[Madison Bumgarner, J.B. Wendelken, Óliver Pér...","[Josiah Gray, Sean Doolittle, Steve Cishek, Ky...",NaN
1,2022-04-19,Arizona at Washington,AZ,WSH,2,https://www.baseball-reference.com/boxes/WAS/W...,"[Tyler Gilbert, Sean Poppen, Joe Mantiply]","[Joan Adon, Víctor Arano, Kyle Finnegan, Tanne...",NaN
2,2022-04-19,San Francisco at NY Mets,SF,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,"[Alex Cobb, Dominic Leone, José Álvarez, Jake ...","[Tylor Megill, Joely Rodríguez, Seth Lugo, Edw...",NaN
3,2022-04-19,San Francisco at NY Mets,SF,NYM,2,https://www.baseball-reference.com/boxes/NYN/N...,"[Logan Webb, Sam Long, Zack Littell, John Breb...","[Max Scherzer, Drew Smith, Trevor May]",NaN
4,2022-04-20,Chicago Sox at Cleveland,CWS,CLE,1,https://www.baseball-reference.com/boxes/CLE/C...,"[Dallas Keuchel, Tanner Banks, Matt Foster, An...","[Shane Bieber, Bryan Shaw, Enyel De Los Santos...",NaN
5,2022-04-20,Chicago Sox at Cleveland,CWS,CLE,2,https://www.baseball-reference.com/boxes/CLE/C...,"[Jimmy Lambert, Reynaldo López, Bennett Sousa,...","[Triston McKenzie, Anthony Gose, Nick Sandlin,...",NaN
6,2022-04-23,Colorado at Detroit,COL,DET,1,https://www.baseball-reference.com/boxes/DET/D...,"[Antonio Senzatela, Ty Blach, Lucas Gilbreath,...","[Tarik Skubal, Wily Peralta, Ángel De Jesús]",NaN
7,2022-04-23,Colorado at Detroit,COL,DET,2,https://www.baseball-reference.com/boxes/DET/D...,"[Austin Gomber, Robert Stephenson, Tyler Kinle...","[Beau Brieske, Alex Lange, Will Vest, Drew Hut...",NaN
8,2022-05-03,Atlanta at NY Mets,ATL,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,"[Charlie Morton, Jesse Chavez]","[David Peterson, Adam Ottavino, Drew Smith, Ed...",NaN
9,2022-05-03,Atlanta at NY Mets,ATL,NYM,2,https://www.baseball-reference.com/boxes/NYN/N...,"[Kyle Wright, Will Smith]","[Carlos Carrasco, Seth Lugo]",NaN


### Dropping Error Column for 2022

**TODO**: Figure out why error column is not in the other ones

In [20]:
pitchers_22 = pitchers_22.drop('error', axis=1)
pitchers_22.head(10)

,date,teams,away_team,home_team,game_number,bref_url,away_pitchers,home_pitchers
0,2022-04-19,Arizona at Washington,AZ,WSH,1,https://www.baseball-reference.com/boxes/WAS/W...,"[Madison Bumgarner, J.B. Wendelken, Óliver Pér...","[Josiah Gray, Sean Doolittle, Steve Cishek, Ky..."
1,2022-04-19,Arizona at Washington,AZ,WSH,2,https://www.baseball-reference.com/boxes/WAS/W...,"[Tyler Gilbert, Sean Poppen, Joe Mantiply]","[Joan Adon, Víctor Arano, Kyle Finnegan, Tanne..."
2,2022-04-19,San Francisco at NY Mets,SF,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,"[Alex Cobb, Dominic Leone, José Álvarez, Jake ...","[Tylor Megill, Joely Rodríguez, Seth Lugo, Edw..."
3,2022-04-19,San Francisco at NY Mets,SF,NYM,2,https://www.baseball-reference.com/boxes/NYN/N...,"[Logan Webb, Sam Long, Zack Littell, John Breb...","[Max Scherzer, Drew Smith, Trevor May]"
4,2022-04-20,Chicago Sox at Cleveland,CWS,CLE,1,https://www.baseball-reference.com/boxes/CLE/C...,"[Dallas Keuchel, Tanner Banks, Matt Foster, An...","[Shane Bieber, Bryan Shaw, Enyel De Los Santos..."
5,2022-04-20,Chicago Sox at Cleveland,CWS,CLE,2,https://www.baseball-reference.com/boxes/CLE/C...,"[Jimmy Lambert, Reynaldo López, Bennett Sousa,...","[Triston McKenzie, Anthony Gose, Nick Sandlin,..."
6,2022-04-23,Colorado at Detroit,COL,DET,1,https://www.baseball-reference.com/boxes/DET/D...,"[Antonio Senzatela, Ty Blach, Lucas Gilbreath,...","[Tarik Skubal, Wily Peralta, Ángel De Jesús]"
7,2022-04-23,Colorado at Detroit,COL,DET,2,https://www.baseball-reference.com/boxes/DET/D...,"[Austin Gomber, Robert Stephenson, Tyler Kinle...","[Beau Brieske, Alex Lange, Will Vest, Drew Hut..."
8,2022-05-03,Atlanta at NY Mets,ATL,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,"[Charlie Morton, Jesse Chavez]","[David Peterson, Adam Ottavino, Drew Smith, Ed..."
9,2022-05-03,Atlanta at NY Mets,ATL,NYM,2,https://www.baseball-reference.com/boxes/NYN/N...,"[Kyle Wright, Will Smith]","[Carlos Carrasco, Seth Lugo]"


### Individual Rows for Pitchers

In [21]:
def explode_pitchers_long(
    df: pd.DataFrame,
    away_col: str = "away_pitchers",
    home_col: str = "home_pitchers",
    pitcher_col: str = "pitcher",
    side_col: str = "side",
    side_labels: tuple[str, str] = ("away", "home"),
    drop_empty: bool = False,   # default: keep everything (use NA for missing)
    sort_output: bool = True,   # sort by date/game_number/side
    date_col: str = "date",
    game_number_col: str = "game_number",
) -> pd.DataFrame:
    """
    Convert game-level pitcher lists (or comma-separated strings) into long format:
    one pitcher per row + an indicator for away/home, with all other columns repeated.

    - Supports list/tuple cells (your current format) and comma-separated strings.
    - If drop_empty=False, missing pitcher lists become one row with pitcher=<NA>.
    - Sorting (optional): date, game_number, then side (away then home).
    """

    def split_pitchers(s):
        # If it's already a list/tuple, clean it directly (avoid pd.isna on lists)
        if isinstance(s, (list, tuple)):
            lst = [str(p).strip() for p in s if p is not None and str(p).strip() != ""]
            return lst if lst else ([] if drop_empty else [pd.NA])

        # Scalar missing
        if s is None or (isinstance(s, float) and pd.isna(s)):
            return [] if drop_empty else [pd.NA]

        s_str = str(s).strip()
        if s_str == "":
            return [] if drop_empty else [pd.NA]

        # Fallback: comma-separated string
        lst = [p.strip() for p in s_str.split(",") if p.strip()]
        return lst if lst else ([] if drop_empty else [pd.NA])

    game_cols = [c for c in df.columns if c not in [away_col, home_col]]

    away_long = (
        df[game_cols + [away_col]]
        .assign(**{
            side_col: side_labels[0],
            pitcher_col: lambda d: d[away_col].map(split_pitchers)
        })
        .drop(columns=[away_col])
        .explode(pitcher_col)
    )

    home_long = (
        df[game_cols + [home_col]]
        .assign(**{
            side_col: side_labels[1],
            pitcher_col: lambda d: d[home_col].map(split_pitchers)
        })
        .drop(columns=[home_col])
        .explode(pitcher_col)
    )

    out = pd.concat([away_long, home_long], ignore_index=True)

    if drop_empty:
        out[pitcher_col] = out[pitcher_col].astype("string")
        out = out.dropna(subset=[pitcher_col])
        out = out[out[pitcher_col].str.strip() != ""]

    if sort_output:
        side_order = {side_labels[0]: 0, side_labels[1]: 1}
        out["_side_order"] = out[side_col].map(side_order)

        sort_cols = []
        if date_col in out.columns:
            sort_cols.append(date_col)
        if game_number_col in out.columns:
            sort_cols.append(game_number_col)
        sort_cols.append("_side_order")

        out = out.sort_values(sort_cols, kind="mergesort").drop(columns=["_side_order"])

    return out.reset_index(drop=True)

In [22]:
pitchers_25_long = explode_pitchers_long(pitchers_25)
pitchers_24_long = explode_pitchers_long(pitchers_24)
pitchers_23_long = explode_pitchers_long(pitchers_23)
pitchers_22_long = explode_pitchers_long(pitchers_22)

display(HTML("<h4>Season 2025</h4>")); display(pitchers_25_long.head(10))
display(HTML("<h4>Season 2024</h4>")); display(pitchers_24_long.head(10))
display(HTML("<h4>Season 2023</h4>")); display(pitchers_23_long.head(10))
display(HTML("<h4>Season 2022</h4>")); display(pitchers_22_long.head(10))

,date,teams,away_team,home_team,game_number,bref_url,side,pitcher
0,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,away,Andre Pallante
1,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,away,Kyle Leahy
2,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,away,JoJo Romero
3,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,away,Phil Maton
4,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,away,Ryan Helsley
5,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,away,Ryan Fernandez
6,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,home,Sean Newcomb
7,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,home,Greg Weissert
8,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,home,Justin Wilson
9,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,home,Justin Slaten


,date,teams,away_team,home_team,game_number,bref_url,side,pitcher
0,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,Casey Mize
1,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,Joey Wentz
2,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,Alex Lange
3,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,Andrew Chafin
4,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,Jason Foley
5,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,Shelby Miller
6,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,home,Adrian Houser
7,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,home,Brooks Raley
8,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,home,Drew Smith
9,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,home,Jake Diekman


,date,teams,away_team,home_team,game_number,bref_url,side,pitcher
0,2023-04-18,Cleveland at Detroit,CLE,DET,1,https://www.baseball-reference.com/boxes/DET/D...,away,Hunter Gaddis
1,2023-04-18,Cleveland at Detroit,CLE,DET,1,https://www.baseball-reference.com/boxes/DET/D...,away,Eli Morgan
2,2023-04-18,Cleveland at Detroit,CLE,DET,1,https://www.baseball-reference.com/boxes/DET/D...,away,Nick Sandlin
3,2023-04-18,Cleveland at Detroit,CLE,DET,1,https://www.baseball-reference.com/boxes/DET/D...,away,James Karinchak
4,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,1,https://www.baseball-reference.com/boxes/CHA/C...,away,Zack Wheeler
5,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,1,https://www.baseball-reference.com/boxes/CHA/C...,away,Gregory Soto
6,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,1,https://www.baseball-reference.com/boxes/CHA/C...,away,Craig Kimbrel
7,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,1,https://www.baseball-reference.com/boxes/CHA/C...,away,Seranthony Domínguez
8,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,1,https://www.baseball-reference.com/boxes/CHA/C...,away,José Alvarado
9,2023-04-18,Cleveland at Detroit,CLE,DET,1,https://www.baseball-reference.com/boxes/DET/D...,home,Matthew Boyd


,date,teams,away_team,home_team,game_number,bref_url,side,pitcher
0,2022-04-19,Arizona at Washington,AZ,WSH,1,https://www.baseball-reference.com/boxes/WAS/W...,away,Madison Bumgarner
1,2022-04-19,Arizona at Washington,AZ,WSH,1,https://www.baseball-reference.com/boxes/WAS/W...,away,J.B. Wendelken
2,2022-04-19,Arizona at Washington,AZ,WSH,1,https://www.baseball-reference.com/boxes/WAS/W...,away,Óliver Pérez
3,2022-04-19,Arizona at Washington,AZ,WSH,1,https://www.baseball-reference.com/boxes/WAS/W...,away,Matt Peacock
4,2022-04-19,San Francisco at NY Mets,SF,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,Alex Cobb
5,2022-04-19,San Francisco at NY Mets,SF,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,Dominic Leone
6,2022-04-19,San Francisco at NY Mets,SF,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,José Álvarez
7,2022-04-19,San Francisco at NY Mets,SF,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,Jake McGee
8,2022-04-19,San Francisco at NY Mets,SF,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,Tyler Rogers
9,2022-04-19,San Francisco at NY Mets,SF,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,Camilo Doval


### Modifying Pitcher Name

In [23]:
def first_last_to_last_first(name):
    if pd.isna(name):
        return name
    s = str(name).strip()
    if s == "" or s.lower() == "nan":
        return pd.NA
    parts = s.split()
    if len(parts) == 1:
        return s  # e.g., "Cher" (rare, but safe)
    last = parts[-1]
    first = " ".join(parts[:-1])
    return f"{last}, {first}"

In [24]:
pitchers_25_long["pitcher"] = pitchers_25_long["pitcher"].map(first_last_to_last_first)
pitchers_24_long["pitcher"] = pitchers_24_long["pitcher"].map(first_last_to_last_first)
pitchers_23_long["pitcher"] = pitchers_23_long["pitcher"].map(first_last_to_last_first)
pitchers_22_long["pitcher"] = pitchers_22_long["pitcher"].map(first_last_to_last_first)

display(HTML("<h4>Season 2025</h4>")); display(pitchers_25_long.head(10))
display(HTML("<h4>Season 2024</h4>")); display(pitchers_24_long.head(10))
display(HTML("<h4>Season 2023</h4>")); display(pitchers_23_long.head(10))
display(HTML("<h4>Season 2022</h4>")); display(pitchers_22_long.head(10))


,date,teams,away_team,home_team,game_number,bref_url,side,pitcher
0,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,away,"Pallante, Andre"
1,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,away,"Leahy, Kyle"
2,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,away,"Romero, JoJo"
3,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,away,"Maton, Phil"
4,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,away,"Helsley, Ryan"
5,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,away,"Fernandez, Ryan"
6,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,home,"Newcomb, Sean"
7,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,home,"Weissert, Greg"
8,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,home,"Wilson, Justin"
9,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,home,"Slaten, Justin"


,date,teams,away_team,home_team,game_number,bref_url,side,pitcher
0,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Mize, Casey"
1,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Wentz, Joey"
2,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Lange, Alex"
3,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Chafin, Andrew"
4,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Foley, Jason"
5,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Miller, Shelby"
6,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,home,"Houser, Adrian"
7,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,home,"Raley, Brooks"
8,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,home,"Smith, Drew"
9,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,home,"Diekman, Jake"


,date,teams,away_team,home_team,game_number,bref_url,side,pitcher
0,2023-04-18,Cleveland at Detroit,CLE,DET,1,https://www.baseball-reference.com/boxes/DET/D...,away,"Gaddis, Hunter"
1,2023-04-18,Cleveland at Detroit,CLE,DET,1,https://www.baseball-reference.com/boxes/DET/D...,away,"Morgan, Eli"
2,2023-04-18,Cleveland at Detroit,CLE,DET,1,https://www.baseball-reference.com/boxes/DET/D...,away,"Sandlin, Nick"
3,2023-04-18,Cleveland at Detroit,CLE,DET,1,https://www.baseball-reference.com/boxes/DET/D...,away,"Karinchak, James"
4,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,1,https://www.baseball-reference.com/boxes/CHA/C...,away,"Wheeler, Zack"
5,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,1,https://www.baseball-reference.com/boxes/CHA/C...,away,"Soto, Gregory"
6,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,1,https://www.baseball-reference.com/boxes/CHA/C...,away,"Kimbrel, Craig"
7,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,1,https://www.baseball-reference.com/boxes/CHA/C...,away,"Domínguez, Seranthony"
8,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,1,https://www.baseball-reference.com/boxes/CHA/C...,away,"Alvarado, José"
9,2023-04-18,Cleveland at Detroit,CLE,DET,1,https://www.baseball-reference.com/boxes/DET/D...,home,"Boyd, Matthew"


,date,teams,away_team,home_team,game_number,bref_url,side,pitcher
0,2022-04-19,Arizona at Washington,AZ,WSH,1,https://www.baseball-reference.com/boxes/WAS/W...,away,"Bumgarner, Madison"
1,2022-04-19,Arizona at Washington,AZ,WSH,1,https://www.baseball-reference.com/boxes/WAS/W...,away,"Wendelken, J.B."
2,2022-04-19,Arizona at Washington,AZ,WSH,1,https://www.baseball-reference.com/boxes/WAS/W...,away,"Pérez, Óliver"
3,2022-04-19,Arizona at Washington,AZ,WSH,1,https://www.baseball-reference.com/boxes/WAS/W...,away,"Peacock, Matt"
4,2022-04-19,San Francisco at NY Mets,SF,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Cobb, Alex"
5,2022-04-19,San Francisco at NY Mets,SF,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Leone, Dominic"
6,2022-04-19,San Francisco at NY Mets,SF,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Álvarez, José"
7,2022-04-19,San Francisco at NY Mets,SF,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"McGee, Jake"
8,2022-04-19,San Francisco at NY Mets,SF,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Rogers, Tyler"
9,2022-04-19,San Francisco at NY Mets,SF,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Doval, Camilo"


### Game ID

In [27]:
def add_game_id(df):
    df = df.copy()

    # YYYYMMDD
    df["game_date_temp"] = pd.to_datetime(df["date"]).dt.strftime("%Y%m%d")

    # clean pitcher name:
    # - remove commas
    # - collapse whitespace
    # - replace spaces with underscores
    df["pitcher_id"] = (
        df["pitcher"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
        .str.replace(r"\s+", "_", regex=True)
    )

    df["game_id"] = (
        df["game_date_temp"]
        + "_"
        + df["away_team"].astype(str)
        + "@"
        + df["home_team"].astype(str)
        + "_"
        + df["pitcher_id"]
    )

    return df.drop(columns=["game_date_temp", "pitcher_id"])


In [28]:
pitchers_25_long = add_game_id(pitchers_25_long)
pitchers_24_long = add_game_id(pitchers_24_long)
pitchers_23_long = add_game_id(pitchers_23_long)
pitchers_22_long = add_game_id(pitchers_22_long)

display(HTML("<h4>Season 2025</h4>")); display(pitchers_25_long.head(10))
display(HTML("<h4>Season 2024</h4>")); display(pitchers_24_long.head(10))
display(HTML("<h4>Season 2023</h4>")); display(pitchers_23_long.head(10))
display(HTML("<h4>Season 2022</h4>")); display(pitchers_22_long.head(10))



,date,teams,away_team,home_team,game_number,bref_url,side,pitcher,game_id
0,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,away,"Pallante, Andre",20250406_STL@BOS_Pallante_Andre
1,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,away,"Leahy, Kyle",20250406_STL@BOS_Leahy_Kyle
2,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,away,"Romero, JoJo",20250406_STL@BOS_Romero_JoJo
3,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,away,"Maton, Phil",20250406_STL@BOS_Maton_Phil
4,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,away,"Helsley, Ryan",20250406_STL@BOS_Helsley_Ryan
5,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,away,"Fernandez, Ryan",20250406_STL@BOS_Fernandez_Ryan
6,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,home,"Newcomb, Sean",20250406_STL@BOS_Newcomb_Sean
7,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,home,"Weissert, Greg",20250406_STL@BOS_Weissert_Greg
8,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,home,"Wilson, Justin",20250406_STL@BOS_Wilson_Justin
9,2025-04-06,St. Louis at Boston,STL,BOS,1,https://www.baseball-reference.com/boxes/BOS/B...,home,"Slaten, Justin",20250406_STL@BOS_Slaten_Justin


,date,teams,away_team,home_team,game_number,bref_url,side,pitcher,game_id
0,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Mize, Casey",20240404_DET@NYM_Mize_Casey
1,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Wentz, Joey",20240404_DET@NYM_Wentz_Joey
2,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Lange, Alex",20240404_DET@NYM_Lange_Alex
3,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Chafin, Andrew",20240404_DET@NYM_Chafin_Andrew
4,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Foley, Jason",20240404_DET@NYM_Foley_Jason
5,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Miller, Shelby",20240404_DET@NYM_Miller_Shelby
6,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,home,"Houser, Adrian",20240404_DET@NYM_Houser_Adrian
7,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,home,"Raley, Brooks",20240404_DET@NYM_Raley_Brooks
8,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,home,"Smith, Drew",20240404_DET@NYM_Smith_Drew
9,2024-04-04,Detroit at NY Mets,DET,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,home,"Diekman, Jake",20240404_DET@NYM_Diekman_Jake


,date,teams,away_team,home_team,game_number,bref_url,side,pitcher,game_id
0,2023-04-18,Cleveland at Detroit,CLE,DET,1,https://www.baseball-reference.com/boxes/DET/D...,away,"Gaddis, Hunter",20230418_CLE@DET_Gaddis_Hunter
1,2023-04-18,Cleveland at Detroit,CLE,DET,1,https://www.baseball-reference.com/boxes/DET/D...,away,"Morgan, Eli",20230418_CLE@DET_Morgan_Eli
2,2023-04-18,Cleveland at Detroit,CLE,DET,1,https://www.baseball-reference.com/boxes/DET/D...,away,"Sandlin, Nick",20230418_CLE@DET_Sandlin_Nick
3,2023-04-18,Cleveland at Detroit,CLE,DET,1,https://www.baseball-reference.com/boxes/DET/D...,away,"Karinchak, James",20230418_CLE@DET_Karinchak_James
4,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,1,https://www.baseball-reference.com/boxes/CHA/C...,away,"Wheeler, Zack",20230418_PHI@CWS_Wheeler_Zack
5,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,1,https://www.baseball-reference.com/boxes/CHA/C...,away,"Soto, Gregory",20230418_PHI@CWS_Soto_Gregory
6,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,1,https://www.baseball-reference.com/boxes/CHA/C...,away,"Kimbrel, Craig",20230418_PHI@CWS_Kimbrel_Craig
7,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,1,https://www.baseball-reference.com/boxes/CHA/C...,away,"Domínguez, Seranthony",20230418_PHI@CWS_Domínguez_Seranthony
8,2023-04-18,Philadelphia at Chicago Sox,PHI,CWS,1,https://www.baseball-reference.com/boxes/CHA/C...,away,"Alvarado, José",20230418_PHI@CWS_Alvarado_José
9,2023-04-18,Cleveland at Detroit,CLE,DET,1,https://www.baseball-reference.com/boxes/DET/D...,home,"Boyd, Matthew",20230418_CLE@DET_Boyd_Matthew


,date,teams,away_team,home_team,game_number,bref_url,side,pitcher,game_id
0,2022-04-19,Arizona at Washington,AZ,WSH,1,https://www.baseball-reference.com/boxes/WAS/W...,away,"Bumgarner, Madison",20220419_AZ@WSH_Bumgarner_Madison
1,2022-04-19,Arizona at Washington,AZ,WSH,1,https://www.baseball-reference.com/boxes/WAS/W...,away,"Wendelken, J.B.",20220419_AZ@WSH_Wendelken_J.B.
2,2022-04-19,Arizona at Washington,AZ,WSH,1,https://www.baseball-reference.com/boxes/WAS/W...,away,"Pérez, Óliver",20220419_AZ@WSH_Pérez_Óliver
3,2022-04-19,Arizona at Washington,AZ,WSH,1,https://www.baseball-reference.com/boxes/WAS/W...,away,"Peacock, Matt",20220419_AZ@WSH_Peacock_Matt
4,2022-04-19,San Francisco at NY Mets,SF,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Cobb, Alex",20220419_SF@NYM_Cobb_Alex
5,2022-04-19,San Francisco at NY Mets,SF,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Leone, Dominic",20220419_SF@NYM_Leone_Dominic
6,2022-04-19,San Francisco at NY Mets,SF,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Álvarez, José",20220419_SF@NYM_Álvarez_José
7,2022-04-19,San Francisco at NY Mets,SF,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"McGee, Jake",20220419_SF@NYM_McGee_Jake
8,2022-04-19,San Francisco at NY Mets,SF,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Rogers, Tyler",20220419_SF@NYM_Rogers_Tyler
9,2022-04-19,San Francisco at NY Mets,SF,NYM,1,https://www.baseball-reference.com/boxes/NYN/N...,away,"Doval, Camilo",20220419_SF@NYM_Doval_Camilo


## Exporting Data

In [29]:
# Collect existing LONG pitchers DataFrames into a dict automatically
double_headers = {
    year: df for year in range(2022, 2026)
    if (df := globals().get(f"pitchers_{str(year)[-2:]}_long")) is not None
}

# Create subfolder: data/double_headers
outdir = Path("data") / "double_headers"
outdir.mkdir(parents=True, exist_ok=True)

for year, df in double_headers.items():
    df.to_csv(outdir / f"double_headers_{year}.csv", index=False, encoding="utf-8")
    print(f"[saved] {outdir / f'double_headers_{year}.csv'}")


[saved] data/double_headers/double_headers_2022.csv
[saved] data/double_headers/double_headers_2023.csv
[saved] data/double_headers/double_headers_2024.csv
[saved] data/double_headers/double_headers_2025.csv
